# 🌱 Project 8 — Find Areas with Healthy Vegetation

## 🎯 Goal

In this project, you will learn how to identify areas with **healthy vegetation** using NDVI.

You will create:

-  An NDVI map.
-  A map showing only healthy vegetation.

---

# 📚 Skills You'll Learn

-  Thresholding
-  Boolean images
-  `gt()` (greater than)
-  `updateMask()`
-  Simple image classification

---

# 🌍 Why Is This Useful?

Imagine a farmer wants to know:

- 🌾 Which fields are healthy?
- 🍂 Which areas are stressed?
- 🌱 Where is vegetation growing well?

Instead of examining the entire NDVI map, we can create a **vegetation mask** that highlights only healthy vegetation.

This approach is widely used in:

- 🌾 Crop monitoring
- 🌲 Forest health assessment
- 🌵 Drought monitoring
- 🏜️ Land degradation studies
- 🌍 Environmental monitoring

---

# 🌱 New Concept: Thresholding

Thresholding means choosing a value and keeping only pixels that satisfy a condition.

Example NDVI values:

| Pixel | NDVI |
|------|------|
| A | 0.1 |
| B | 0.3 |
| C | 0.7 |
| D | 0.8 |

Suppose we define:

```text
Healthy vegetation = NDVI > 0.5
```

The result becomes:

| Pixel | Keep? |
|------|------|
| A | ❌ |
| B | ❌ |
| C | ✅ |
| D | ✅ |

Everything below the threshold is ignored.

---

# ⭐ New Function: `gt()`

Today you'll learn:

```javascript
.gt()
```

It means:

> **Greater Than**

Example:

```javascript
ndvi.gt(0.5)
```

asks:

> "Is each pixel greater than 0.5?"

The answer for every pixel is either:

- ✅ True
- ❌ False

---

# Step 1 — Create a Study Area

```javascript
var cityArea = ee.Geometry.Polygon([
  [
    [51.55,32.55],
    [51.80,32.55],
    [51.80,32.75],
    [51.55,32.75],
    [51.55,32.55]
  ]
]);
```

Nothing new here.

---

# Step 2 — Load a Sentinel-2 Image

```javascript
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(cityArea)
.filterDate('2025-06-01','2025-08-31')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(cityArea);
```

We use a summer image because vegetation is usually at its healthiest during this season.

---

# Step 3 — Calculate NDVI

```javascript
var ndvi = image
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');
```

Remember what happens:

```text
Sentinel-2 Image

↓

Calculate NDVI

↓

NDVI Image
```

The result is still an **`ee.Image`**, but each pixel now contains an NDVI value.

---

# Step 4 — Display the NDVI Map

```javascript
var ndviVis = {
  min:-1,
  max:1,
  palette:[
    'brown',
    'yellow',
    'green'
  ]
};

Map.centerObject(cityArea,10);

Map.addLayer(
  ndvi,
  ndviVis,
  'NDVI'
);
```

Brown represents low vegetation, while green represents healthy vegetation.

---

# Step 5 — Create a Healthy Vegetation Mask ⭐

This is the new part.

```javascript
var healthyVegetation = ndvi.gt(0.5);
```

Let's understand what happens.

Before:

```text
NDVI Image

0.1
0.3
0.7
0.8
```

After:

```text
NDVI > 0.5

False
False
True
True
```

The result is **not another NDVI map**.

Instead, it is a **Boolean image**.

Each pixel stores only:

```text
True

or

False
```

Think of it as a stencil:

```text
✔ Keep

✖ Hide

✔ Keep

✔ Keep
```

---

# Step 6 — Apply the Mask

Now we use what you learned in Project 4.

```javascript
var vegetationMap = ndvi.updateMask(
  healthyVegetation
);
```

Think of the process like this:

```text
NDVI Image

        +

Boolean Mask

        ↓

Healthy Vegetation Only
```

Pixels where the mask is **False** become invisible.

Pixels where the mask is **True** remain visible.

---

# Step 7 — Display Healthy Vegetation

```javascript
Map.addLayer(
  vegetationMap,
  {
    palette:['green']
  },
  'Healthy Vegetation'
);
```

Now only pixels with:

```text
NDVI > 0.5
```

are displayed.

---

# Complete Code

```javascript
// Study Area
var cityArea = ee.Geometry.Polygon([
  [
    [51.55,32.55],
    [51.80,32.55],
    [51.80,32.75],
    [51.55,32.75],
    [51.55,32.55]
  ]
]);

// Load Image
var image = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(cityArea)
.filterDate('2025-06-01','2025-08-31')
.sort('CLOUDY_PIXEL_PERCENTAGE')
.first()
.clip(cityArea);

// Calculate NDVI
var ndvi = image
  .normalizedDifference(['B8','B4'])
  .rename('NDVI');

// Visualization
var ndviVis = {
  min:-1,
  max:1,
  palette:[
    'brown',
    'yellow',
    'green'
  ]
};

Map.centerObject(cityArea,10);

Map.addLayer(
  ndvi,
  ndviVis,
  'NDVI'
);

// Create threshold
var healthyVegetation = ndvi.gt(0.5);

// Apply mask
var vegetationMap = ndvi.updateMask(
  healthyVegetation
);

// Display result
Map.addLayer(
  vegetationMap,
  {
    palette:['green']
  },
  'Healthy Vegetation'
);
```

---

# Workflow Summary

```text
Sentinel-2 Image

        ↓

Calculate NDVI

        ↓

NDVI Image

        ↓

Apply Threshold
(NDVI > 0.5)

        ↓

Boolean Mask
(True / False)

        ↓

updateMask()

        ↓

Healthy Vegetation Map
```

---

# Key Concepts

## ✅ `gt()`

Think:

```javascript
ndvi.gt(0.5)
```

means:

> "Give me pixels where NDVI is greater than 0.5."

Other useful comparison functions:

| Function | Meaning |
|----------|---------|
| `.gt()` | Greater than |
| `.gte()` | Greater than or equal |
| `.lt()` | Less than |
| `.lte()` | Less than or equal |
| `.eq()` | Equal |

---

## ✅ Thresholding Creates a Mask

Before:

```text
NDVI

0.2
0.6
0.8
```

After:

```text
NDVI > 0.5

False
True
True
```

The output is a **Boolean image**, not a classified map.

---

## ✅ Image + Rule = New Map

A common pattern in Earth Engine is:

```text
Image

      +

Rule

      ↓

New Map
```

Examples:

Healthy vegetation:

```javascript
ndvi.gt(0.5)
```

Water:

```javascript
ndwi.gt(0)
```

Burned areas:

```javascript
nbr.lt(0.2)
```

---

# Common Beginner Mistakes

## ❌ Mistake 1 — Using JavaScript Comparisons

Wrong:

```javascript
ndvi > 0.5
```

This compares objects, not image pixels.

Earth Engine requires:

```javascript
ndvi.gt(0.5)
```

which evaluates every pixel individually.

---

## ❌ Mistake 2 — Confusing the Mask with the Final Map

This:

```javascript
var mask = ndvi.gt(0.5);
```

creates only a **Boolean image**.

To display the actual NDVI values for healthy vegetation, you still need:

```javascript
var vegetation = ndvi.updateMask(mask);
```

---

## ❌ Mistake 3 — Assuming the Threshold Is Universal

A threshold of **0.5** works well in many situations, but it is **not a universal rule**.

Different vegetation types, climates, and study objectives may require different threshold values.

---

# Practice Exercise

Create a healthy vegetation map for another region.

Requirements:

- ✅ Create a polygon study area.
- ✅ Load a Sentinel-2 image.
- ✅ Calculate NDVI.
- ✅ Create a threshold:
  - Healthy vegetation = **NDVI > 0.4**
- ✅ Apply the mask.
- ✅ Display:
  - NDVI map
  - Healthy vegetation map

---

# 🧠 Check Your Understanding

Before writing the code, answer these questions.

### 1. Why do we use `.gt()` instead of writing:

```javascript
ndvi > 0.4
```

Explain the difference between JavaScript comparisons and Earth Engine image operations.

---

### 2. What is the difference between:

```javascript
var mask = ndvi.gt(0.4);
```

and

```javascript
var vegetation = ndvi.updateMask(mask);
```

Explain what each variable contains and why both are needed.